In [ ]:
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
from config import Config as cfg
import json
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

In [ ]:
model_name = cfg.BASE_MODEL_NAME

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
#load CNN articles data
raw_data_path = cfg.Raw_DATA_PATH
with open(raw_data_path,'r') as f:
  article_data = json.load(f)

In [ ]:
data = [i['text'] for i in article_data]

In [ ]:
def make_message(data):
  #making data into the format that model takes (chat like format)
  one_shot_user = {
        'role': 'user',
        'content': "Summarize the following text:\n\nArtificial intelligence is rapidly transforming industries such as healthcare, finance, and transportation. Machine learning models are being used to improve decision-making, automate tasks, and enhance user experiences. However, concerns about ethics, bias, and job displacement remain important challenges."
    }

  one_shot_assistant = {
        'role': 'assistant',
        'content': """Subject: Artificial Intelligence

    Keywords: AI, machine learning, ethics

    Summary: Artificial intelligence is transforming multiple industries by improving decision-making and automation, but raises concerns about ethics, bias, and job displacement."""
    }

  # data -> one shot message form
  message = [
    {'role': 'system', 'content': 'you need to summarize the text with three keywords and summary. the format has to contain Subject: , Keywords: and Summary: parts.'},
    one_shot_user,
    one_shot_assistant,
    {'role': 'user', 'content':f'Summarize the following text: {data}'},
  ]

  return message


In [ ]:
#Temperature=0.7, TopP=0.8, TopK=20, and MinP=0 -> recommended config in non-thinking mode by the supplier
from transformers import GenerationConfig
generation_config = GenerationConfig(
    temperature = 0.7,
    top_p = 0.8,
    top_k = 20,
    min_p = 0,
    max_new_tokens = 500)

In [ ]:
def model_output(tokenizer, data):
  # A function model generates
  #Qwen3 4b model has thinking part so have to remove thinking parts when we save model generation
  #article -> message with one shot -> chat templated -> tokenize -> generate
    message = make_message(data)
    chat_template_data = tokenizer.apply_chat_template(message, tokenize = False, truncation = True, enable_thinking = False, add_generation_prompt = True)
    model_inputs = tokenizer([chat_template_data], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            generation_config=generation_config,
        )
    output_ids = generated_ids[0].cpu().tolist()
    # remove prompt
    index = len(output_ids) - output_ids[::-1].index(151668)

    # decode
    content = tokenizer.decode(
        generated_ids[0][index:].cpu(),
        skip_special_tokens=True
    ).strip()

    return content


In [ ]:
def split_dataset(dataset, num_part):
    """
    A function splits datasets to train
    dataset: list[str]
    num_part: int (num of chunks)

    return: list[list[str]]
    """

    if num_part <= 0:
        raise ValueError("num_part must be positive")

    return [
        dataset[i:i + num_part]
        for i in range(0, len(dataset), num_part)
    ]


In [ ]:
import re

def validate_data(data, tokenizer):

    model_output = data

    # --------------------
    # Subject
    # --------------------
    subject_match = re.search(
        r"subject\s*:\s*(.*?)(?:\n|keywords\s*:|summary\s*:|$)",
        data,
        re.I | re.S
    )

    if subject_match:
        subject = subject_match.group(1).strip()
        has_subject = 1
    else:
        subject = ""
        has_subject = 0

    # --------------------
    # Keywords
    # --------------------
    keywords_match = re.search(
        r"keywords\s*:\s*(.*?)(?:\n|summary\s*:|$)",
        data,
        re.I | re.S
    )

    if keywords_match:
        keywords = keywords_match.group(1).strip()
        has_keywords = 1
    else:
        keywords = ""
        has_keywords = 0

    # --------------------
    # Summary
    # --------------------
    summary_match = re.search(
        r"summary\s*:\s*(.*)",
        data,
        re.I | re.S
    )

    if summary_match:
        summary = summary_match.group(1).strip()
        has_summary = 1
        num_summary_tokens = len(tokenizer.encode(summary))
    else:
        summary = ""
        has_summary = 0
        num_summary_tokens = 0

    return [
        model_output,
        has_subject,
        has_keywords,
        has_summary,
        num_summary_tokens,
        subject,
        keywords,
        summary
    ]

In [ ]:
from tqdm import tqdm

def generate_data(tokenizer, dataset, num_chunk):

    # dataset split
    split_datasets = split_dataset(dataset, num_chunk)

    # chunk loop
    for chunk_idx, chunk in enumerate(tqdm(split_datasets, desc="Chunk Progress")):
      #  dict for saving result
        generated_data = {
          'original_article': [],
          "model_output": [],
          'has_subject': [],
          "has_keywords": [],
          "has_summary": [],
          "num_summary_tokens": [],
          'subject': [],
          "keywords": [],
          "summary": [],
          }

        print(f"\n===== Processing Chunk {chunk_idx+1}/{len(split_datasets)} =====")

        # data loop
        for data in tqdm(chunk, desc=f"Chunk {chunk_idx+1}", leave=False):
            #data : original_article
            # model generates data
            output = model_output(tokenizer,data)

            print("Allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
            print("Reserved :", torch.cuda.memory_reserved() / 1024**2, "MB")

            # validation
            model_output, has_subject, has_keywords, has_summary, num_tokens,subject, keywords, summary = validate_data(output, tokenizer)

            # display validation
            print(f"[VALIDATION] keywords:{has_keywords} | summary:{has_summary} | tokens:{num_tokens}")

            # save
            generated_data["original_article"].append(data)
            generated_data["model_output"].append(model_output)
            generated_data["has_subject"].append(has_subject)
            generated_data["has_keywords"].append(has_keywords)
            generated_data["has_summary"].append(has_summary)
            generated_data["num_summary_tokens"].append(num_tokens)
            generated_data["subject"].append(subject)
            generated_data["keywords"].append(keywords)
            generated_data["summary"].append(summary)

        with open(f"generated_chunk_{chunk_idx}.json", "w") as f:
          json.dump(generated_data, f, ensure_ascii=False, indent=2)

        ## this part is for when you control model generation (chunks)
        # user_input = input("Continue to next chunk? (y/n): ")
        # if user_input.lower() != "y":
        #     print("Stopping generation.")
        #     break

    print("\nGeneration finished.")




In [ ]:
#before generate datas some input's tokens are too large for my resources so remove
data_filtered = [
    i for i in data if 100 < len(tokenizer.encode(tokenizer.apply_chat_template(make_message(i),tokenize = False, truncation = False, enable_thinking = False, add_generation_prompt = True))) <4000 ]



In [ ]:
#generate summaries by using model
generate_data(tokenizer,data_filtered,50)